# Practical 07 – Classification II and Evaluation

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = "Alexa_dataset.csv"
LOCAL_CANDIDATES = [Path("data") / DATA_FILE, Path("Lecture07_Classification_II_and_Evaluation") / "data" / DATA_FILE]
GITHUB_URL = "https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture07_Classification_II_and_Evaluation/data/Alexa_dataset.csv"
local_path = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
data_source = local_path if local_path else GITHUB_URL
df = pd.read_csv(data_source)
print(f"Loaded from: {data_source}")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

X = df[["Age", "Minutes of music consumed"]]
y = df["Purchased Alexa"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Naive Bayes": GaussianNB()
}
rows=[]
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred=clf.predict(X_test)
    rows.append({"model":name,"accuracy":accuracy_score(y_test,pred),
                 "precision":precision_score(y_test,pred),
                 "recall":recall_score(y_test,pred),"f1":f1_score(y_test,pred)})
    print(name, "\n", confusion_matrix(y_test,pred))
pd.DataFrame(rows).sort_values("f1",ascending=False)

## Source Chapter Exercise – Read a Confusion Matrix

The author chapter provides this confusion matrix:

```text
[[64, 4],
 [ 3,29]]
```

Assume:

- Class 1 = Purchased Alexa (positive)
- Class 0 = Did Not Purchase (negative)

### Questions

Identify:

- TN
- FP
- FN
- TP
- total predictions
- total mistakes

### Source answer

- TN = 64
- FP = 4
- FN = 3
- TP = 29
- Total predictions = 100
- Total errors = 7

From the same matrix:

- Accuracy = 93 / 100 = **0.93**
- Precision = 29 / 33 ≈ **0.8788**
- Recall = 29 / 32 ≈ **0.9063**

## Exercise
If false negatives are especially costly, which metric should receive more attention?

## Source Chapter Visualization – Decision Tree, Random Forest, and Naïve Bayes

The author chapter plots the classification regions for the three classifiers. The following helper recreates that idea using the same two features.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_regions(model, X_plot, y_plot, title):
    x1_min, x1_max = X_plot.iloc[:, 0].min() - 2, X_plot.iloc[:, 0].max() + 2
    x2_min, x2_max = X_plot.iloc[:, 1].min() - 5000, X_plot.iloc[:, 1].max() + 5000

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 250),
        np.linspace(x2_min, x2_max, 250)
    )

    grid = pd.DataFrame({
        X_plot.columns[0]: xx1.ravel(),
        X_plot.columns[1]: xx2.ravel()
    })

    zz = model.predict(grid).reshape(xx1.shape)

    plt.contourf(xx1, xx2, zz, alpha=0.25)
    plt.scatter(X_plot.iloc[:, 0], X_plot.iloc[:, 1], c=y_plot, edgecolor="k")
    plt.xlabel(X_plot.columns[0])
    plt.ylabel(X_plot.columns[1])
    plt.title(title)
    plt.show()

for name, clf in models.items():
    clf.fit(X_train, y_train)
    plot_regions(clf, X_train, y_train, f"{name} – Training Set")
    plot_regions(clf, X_test, y_test, f"{name} – Test Set")